# 18.1 贝叶斯线性回归 / Bayesian Linear Regression

**中文**：前面学的所有回归/分类都给出一个**点估计**——一组"最好的"参数(一条最佳拟合线)。但它们回答不了一个关键问题:*"你对这个估计有多确定？"* 数据少的地方、外推的地方，模型其实很没底气,却照样给你一个自信满满的预测。**贝叶斯方法**从根本上不同:它不求"一个最优参数",而是求参数的**整个概率分布**——于是每个预测都自带**不确定性**。本部分(Part 18)从贝叶斯线性回归开始,一路到高斯过程和贝叶斯优化。
**English**: Every regression/classification so far gave a **point estimate** — one set of "best" parameters (a single best-fit line). But they can't answer a crucial question: *"how certain are you about that estimate?"* Where data is scarce or you extrapolate, the model is actually clueless yet still hands you a confident prediction. **Bayesian methods** are fundamentally different: instead of "one optimal parameter," they seek the **entire probability distribution** over parameters — so every prediction carries **uncertainty**. This part (Part 18) starts with Bayesian linear regression and goes to Gaussian processes and Bayesian optimization.

---

**中文**：贝叶斯推断的核心就一条公式——**贝叶斯定理**:
**English**: Bayesian inference's core is a single formula — **Bayes' theorem**:

$$\underbrace{p(\theta|\mathcal D)}_{\text{后验 posterior}} = \frac{\overbrace{p(\mathcal D|\theta)}^{\text{似然 likelihood}}\ \overbrace{p(\theta)}^{\text{先验 prior}}}{\underbrace{p(\mathcal D)}_{\text{证据 evidence}}} \ \propto\ p(\mathcal D|\theta)\,p(\theta)$$

**中文**：逐项解释:
**English**: Term by term:
- **先验 $p(\theta)$**:看到数据**之前**，你对参数的信念(比如"权重大概在 0 附近")。
  **Prior $p(\theta)$**: your belief about parameters **before** seeing data (e.g. "weights are probably near 0").
- **似然 $p(\mathcal D|\theta)$**:给定参数，观测到这批数据的概率(就是普通回归的"拟合优度")。
  **Likelihood $p(\mathcal D|\theta)$**: the probability of the observed data given parameters (ordinary regression's "goodness of fit").
- **后验 $p(\theta|\mathcal D)$**:看到数据**之后**，你更新过的信念——这就是我们要求的。
  **Posterior $p(\theta|\mathcal D)$**: your **updated** belief after seeing data — what we want.
- **证据 $p(\mathcal D)$**:归一化常数(常最难算,后面 MCMC/变分就是为绕开它)。
  **Evidence $p(\mathcal D)$**: the normalizing constant (often the hardest to compute; MCMC/VI later exist to bypass it).

**中文**：一句话:**后验 ∝ 似然 × 先验**——用数据把先验信念"更新"成后验信念。数据越多，后验越被似然主导(数据说话);数据越少，先验影响越大(先验兜底)。这天然实现了**正则化**和**不确定性量化**。
**English**: In one line: **posterior ∝ likelihood × prior** — data updates the prior belief into the posterior. More data → the posterior is dominated by the likelihood (data speaks); less data → the prior matters more (the prior holds the fort). This naturally provides **regularization** and **uncertainty quantification**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 贝叶斯必考）**
> **中文**：贝叶斯 = 学参数的**分布**而非点估计。**后验∝似然×先验**。**贝叶斯线性回归**:高斯先验 + 高斯似然 → 后验也是高斯(**共轭 conjugate**, 闭式解, 无需采样)。产出**预测分布**(均值 + 方差), 外推处方差自动变大=知道自己不知道。**vs 频率派(OLS/岭回归)**:①MAP 估计 = 岭回归(高斯先验的负对数=L2 正则), 所以岭回归其实是"偷偷的贝叶斯";②贝叶斯给完整不确定性, 频率派只给点估计+(近似)置信区间。**共轭先验**=先验和后验同族(Beta-Bernoulli, Gamma-Poisson, Normal-Normal), 让后验有闭式解。
> **English**: Bayesian = learn a **distribution** over parameters, not a point estimate. **posterior ∝ likelihood × prior**. **Bayesian linear regression**: Gaussian prior + Gaussian likelihood → Gaussian posterior (**conjugate**, closed form, no sampling). Yields a **predictive distribution** (mean + variance); variance auto-grows where you extrapolate = knowing what you don't know. **vs frequentist (OLS/ridge)**: ① the MAP estimate = ridge regression (a Gaussian prior's negative log = L2 regularization), so ridge is "secretly Bayesian"; ② Bayesian gives full uncertainty, frequentist only a point estimate + (approximate) confidence intervals. **Conjugate prior** = prior and posterior in the same family (Beta-Bernoulli, Gamma-Poisson, Normal-Normal), giving a closed-form posterior.


In [ ]:

# ============================================================
# 数据:一维带噪声的线性关系 / 1D noisy linear data
# 中文:真实关系 y = w0 + w1*x + 噪声。我们故意只在 x 的一小段区间给少量数据,
#      好观察"数据外的地方贝叶斯的不确定性会不会自动变大"。
# English: true relation y = w0 + w1*x + noise. We deliberately give few points over a small x-range,
#      to see whether Bayesian uncertainty automatically grows outside the data.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
np.random.seed(1)
true_w = np.array([-0.3, 0.5])                                # 真实截距和斜率 / true intercept & slope
noise_sigma = 0.2                                             # 观测噪声标准差 / observation noise std
N = 12
X_raw = np.random.uniform(-0.9, -0.1, N)                      # 只在左半段采样(制造外推区)/ data only on left
y = true_w[0] + true_w[1]*X_raw + np.random.normal(0, noise_sigma, N)
Phi = np.c_[np.ones(N), X_raw]                                # 设计矩阵 [1, x] / design matrix

# 普通最小二乘(频率派点估计, 作对比)/ OLS point estimate for comparison
w_ols = np.linalg.lstsq(Phi, y, rcond=None)[0]
print("真实参数 / true w:", true_w)
print("OLS 点估计 / OLS point estimate:", np.round(w_ols,3), "(只给一条线, 不给不确定性)")


**中文**：现在做**贝叶斯**版本。设定:
**English**: Now the **Bayesian** version. Setup:
- **先验** $p(\mathbf w)=\mathcal N(\mathbf 0,\alpha^{-1}I)$:权重先验是均值 0、精度 $\alpha$ 的高斯(相信权重不会太大)。
  **Prior** $p(\mathbf w)=\mathcal N(\mathbf 0,\alpha^{-1}I)$: a zero-mean Gaussian with precision $\alpha$ (weights aren't too large).
- **似然** $p(y|\mathbf w)=\mathcal N(\Phi\mathbf w,\beta^{-1}I)$:观测噪声高斯,精度 $\beta=1/\sigma^2$。
  **Likelihood** $p(y|\mathbf w)=\mathcal N(\Phi\mathbf w,\beta^{-1}I)$: Gaussian noise with precision $\beta=1/\sigma^2$.

**中文**：因为高斯先验遇上高斯似然是**共轭**的，后验也是高斯，有**闭式解**:
**English**: Because a Gaussian prior with a Gaussian likelihood is **conjugate**, the posterior is also Gaussian with a **closed form**:

$$p(\mathbf w|\mathcal D)=\mathcal N(\mathbf m_N,\mathbf S_N),\quad \mathbf S_N^{-1}=\alpha I+\beta\Phi^\top\Phi,\quad \mathbf m_N=\beta\,\mathbf S_N\Phi^\top\mathbf y$$

**中文**：$\mathbf m_N$ 是后验均值(最可能的权重)，$\mathbf S_N$ 是后验协方差(权重的不确定性)。注意 $\mathbf m_N$ 恰好等于**岭回归**的解(正则强度 $\lambda=\alpha/\beta$)——这就是"岭回归=高斯先验下的 MAP"。
**English**: $\mathbf m_N$ is the posterior mean (most likely weights), $\mathbf S_N$ the posterior covariance (weight uncertainty). Note $\mathbf m_N$ equals the **ridge regression** solution (regularization $\lambda=\alpha/\beta$) — "ridge = MAP under a Gaussian prior."


In [ ]:

# ============================================================
# 闭式后验 / closed-form posterior (conjugate)
# ============================================================
alpha = 2.0                                                  # 先验精度(1/先验方差)/ prior precision
beta  = 1.0/noise_sigma**2                                   # 似然精度(1/噪声方差)/ likelihood precision
S0_inv = alpha*np.eye(2)                                     # 先验精度矩阵 / prior precision matrix
SN_inv = S0_inv + beta*Phi.T@Phi                             # 后验精度 = 先验 + 数据贡献 / posterior precision
SN = np.linalg.inv(SN_inv)                                   # 后验协方差 / posterior covariance
mN = beta*SN@Phi.T@y                                         # 后验均值 / posterior mean

print("后验均值 m_N (最可能权重) / posterior mean:", np.round(mN,3))
print("后验标准差 (权重不确定性) / posterior std:", np.round(np.sqrt(np.diag(SN)),3))
# 验证:后验均值 == 岭回归解 (λ=α/β) / verify posterior mean equals ridge solution
lam=alpha/beta; w_ridge=np.linalg.solve(Phi.T@Phi+lam*np.eye(2), Phi.T@y)
print("岭回归解(λ=α/β) / ridge:", np.round(w_ridge,3), "== 后验均值? / == posterior mean?",
      np.allclose(mN,w_ridge))


**中文**：贝叶斯回归最有用的产出是**预测分布**——对一个新的 $x_*$，不只给一个预测值，还给一个**方差**:
**English**: The most useful output of Bayesian regression is the **predictive distribution** — for a new $x_*$, not just a value but also a **variance**:

$$p(y_*|x_*,\mathcal D)=\mathcal N\big(\mathbf m_N^\top\phi(x_*),\ \sigma_N^2(x_*)\big),\quad \sigma_N^2(x_*)=\underbrace{\beta^{-1}}_{\text{噪声}}+\underbrace{\phi(x_*)^\top\mathbf S_N\phi(x_*)}_{\text{参数不确定性}}$$

**中文**：预测方差有两部分:**固有噪声**($\beta^{-1}$，永远存在)+ **参数不确定性**(第二项，离数据越远越大)。关键就在第二项——在**没有数据的地方**，$\phi(x_*)^\top\mathbf S_N\phi(x_*)$ 会自动变大,于是预测的"误差棒"张开。**模型知道自己在外推、知道自己不确定。**
**English**: The predictive variance has two parts: **intrinsic noise** ($\beta^{-1}$, always present) + **parameter uncertainty** (the second term, growing far from data). The key is the second term — **where there is no data**, $\phi(x_*)^\top\mathbf S_N\phi(x_*)$ auto-grows, so the "error bars" widen. **The model knows it is extrapolating and knows it is uncertain.**


In [ ]:

# ============================================================
# 预测分布 + 从后验采样若干条回归线 / predictive distribution + sample lines from posterior
# ============================================================
xs = np.linspace(-2.0, 1.2, 200)                             # 预测范围(超出数据区)/ prediction range (beyond data)
Phi_s = np.c_[np.ones_like(xs), xs]
pred_mean = Phi_s@mN                                         # 预测均值 / predictive mean
pred_var  = 1.0/beta + np.sum(Phi_s@SN*Phi_s, axis=1)        # 预测方差(噪声+参数不确定)/ predictive variance
pred_std  = np.sqrt(pred_var)

# 从后验分布采样 20 组权重, 每组画一条线 —— 直观展示"参数的分布" / sample 20 weight sets from posterior
rng=np.random.default_rng(0)
w_samples = rng.multivariate_normal(mN, SN, size=20)
print("预测区间在数据区内最窄、外推处最宽 / uncertainty is narrowest in-data, widest when extrapolating")
print(f"数据中心 x=-0.5 处预测std {pred_std[np.argmin(abs(xs+0.5))]:.3f}  vs 远处 x=1.0 处 {pred_std[np.argmin(abs(xs-1.0))]:.3f}")


**中文**：可视化——左图画**预测分布**(均值 + 阴影不确定带),右图画**从后验采样的多条线**和**先验 vs 后验**在参数空间的对比。
**English**: Visualization — left: the **predictive distribution** (mean + shaded uncertainty band); right: **lines sampled from the posterior** and the **prior vs posterior** in parameter space.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.8))
# ① 预测分布 / predictive distribution
ax[0].scatter(X_raw,y,c="k",zorder=5,label="数据 data")
ax[0].plot(xs, true_w[0]+true_w[1]*xs, "g--", label="真实 true")
ax[0].plot(xs, pred_mean, "b", label="后验均值 mean")
for k,c in [(1,0.3),(2,0.15)]:
    ax[0].fill_between(xs, pred_mean-k*pred_std, pred_mean+k*pred_std, color="b", alpha=c)
ax[0].axvspan(X_raw.min(),X_raw.max(),color="orange",alpha=0.08)
ax[0].set_title("预测分布:外推处不确定性张开 / uncertainty grows outside data"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].legend(fontsize=8)
# ② 从后验采样的回归线 / lines sampled from posterior
ax[1].scatter(X_raw,y,c="k",zorder=5)
for w in w_samples: ax[1].plot(xs, Phi_s@w, "b", alpha=0.2)
ax[1].plot(xs, true_w[0]+true_w[1]*xs, "g--", lw=2, label="真实 true")
ax[1].set_title("从后验采样的20条线(数据处收紧)/ 20 posterior lines"); ax[1].set_xlabel("x"); ax[1].set_ylabel("y"); ax[1].legend(fontsize=8)
# ③ 参数空间:先验 vs 后验 / prior vs posterior in weight space
gx,gy=np.mgrid[-1:0.5:100j, -0.5:1.5:100j]; pos=np.dstack([gx,gy])
prior=stats.multivariate_normal([0,0], np.eye(2)/alpha).pdf(pos)
post =stats.multivariate_normal(mN, SN).pdf(pos)
ax[2].contour(gx,gy,prior,colors="gray",alpha=0.5,linewidths=1)
ax[2].contour(gx,gy,post,cmap="Blues")
ax[2].plot(*true_w,"g*",ms=15,label="真实 w"); ax[2].plot(*w_ols,"rx",ms=10,label="OLS")
ax[2].set_title("参数空间:先验(灰)→后验(蓝)/ prior→posterior"); ax[2].set_xlabel("w0 截距"); ax[2].set_ylabel("w1 斜率"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/bay01_viz.png",dpi=80); plt.show()
print("后验(蓝)比先验(灰)集中得多, 且把真实值w*(绿星)包在里面")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **贝叶斯给的是"预测 + 不确定性"**:同一条数据,OLS 只给一条线,贝叶斯给一条均值线**加一条随位置变化的误差带**。关键洞察在左图——**在有数据的橙色区间,不确定带很窄;一旦外推到左右两侧没数据的地方,误差带迅速张开**。模型**知道自己在瞎猜**,这在高风险决策(医疗、金融、自动驾驶)里价值千金。
2. **岭回归其实是"偷偷的贝叶斯"**:我们验证了后验均值 $\mathbf m_N$ **精确等于**岭回归解(正则 $\lambda=\alpha/\beta$)。所以你平时用的 L2 正则,数学上就是"给权重加了个均值 0 的高斯先验后求 MAP"。**先验强度 α ↔ 正则强度 λ**。这条对应关系是面试高频考点。
3. **共轭让一切闭式可算**:高斯先验×高斯似然=高斯后验,不用采样、不用优化,一个矩阵求逆就得到完整后验。**代价**:共轭只在特定"先验-似然"搭配下成立(高斯、Beta-伯努利等);一旦模型复杂(如贝叶斯逻辑回归、神经网络),后验就没有闭式解——这正是后面 **Laplace 近似、MCMC、变分推断**要解决的。

**English**:
1. **Bayesian gives "prediction + uncertainty"**: on the same data, OLS gives one line; Bayesian gives a mean line **plus a position-dependent error band**. The key insight is the left plot — **within the orange data region the band is narrow; once you extrapolate to the data-free sides, the band rapidly widens**. The model **knows it is guessing**, which is invaluable in high-stakes decisions (medicine, finance, self-driving).
2. **Ridge is "secretly Bayesian"**: we verified the posterior mean $\mathbf m_N$ **exactly equals** the ridge solution (regularization $\lambda=\alpha/\beta$). So your everyday L2 regularization is mathematically "MAP after placing a zero-mean Gaussian prior on the weights." **Prior strength α ↔ regularization λ.** A frequent interview point.
3. **Conjugacy makes everything closed-form**: Gaussian prior × Gaussian likelihood = Gaussian posterior; no sampling, no optimization — one matrix inverse gives the full posterior. **The cost**: conjugacy holds only for specific prior-likelihood pairs (Gaussian, Beta-Bernoulli, etc.); once the model is complex (Bayesian logistic regression, neural nets) the posterior has no closed form — exactly what **Laplace approximation, MCMC, and variational inference** solve next.

> 💼 **实战视角 / Practical angle**
> **中文**:贝叶斯回归的实战价值:①**小数据 + 需要不确定性**的场景(A/B 早停、传感器融合、金融风险);②**主动学习**(去标注模型最不确定的样本);③**贝叶斯优化**(用不确定性指导下一次实验,18.7);④先验注入领域知识。**权衡**:计算贵(要算/近似后验)、先验选择有主观性(但可用"经验贝叶斯"从数据估先验)。**什么时候不用**:大数据下点估计已足够、且不确定性无关紧要时,普通回归更省。面试金句:*"贝叶斯学参数的分布而非点估计, 天然量化不确定性; 岭回归就是高斯先验下的 MAP; 共轭先验给闭式后验, 否则要 Laplace/MCMC/VI 近似。"*
> **English**: Practical value of Bayesian regression: ① **small data + need uncertainty** (A/B early-stopping, sensor fusion, financial risk); ② **active learning** (label the samples the model is least sure about); ③ **Bayesian optimization** (use uncertainty to guide the next experiment, 18.7); ④ inject domain knowledge via priors. **Trade-offs**: computationally costly (compute/approximate the posterior); prior choice is subjective (though empirical Bayes can estimate priors from data). **When not to**: with big data where a point estimate suffices and uncertainty is irrelevant, plain regression is cheaper. Interview line: *"Bayesian learns a distribution over parameters, not a point estimate, quantifying uncertainty for free; ridge is MAP under a Gaussian prior; conjugate priors give a closed-form posterior, else use Laplace/MCMC/VI."*

---
### 小结 / Summary
- **中文**:贝叶斯 = 后验∝似然×先验, 学参数的分布; 高斯先验+高斯似然→共轭高斯后验(闭式)。
- **English**: Bayesian = posterior ∝ likelihood × prior, learning a distribution; Gaussian prior + Gaussian likelihood → conjugate Gaussian posterior (closed form).
- **中文**:核心价值=预测分布带不确定性, 外推处自动张开; 后验均值=岭回归解(λ=α/β)。
- **English**: Core value = a predictive distribution with uncertainty that auto-widens on extrapolation; posterior mean = ridge solution (λ=α/β).
- **中文**:共轭闭式仅特例; 复杂模型后验无闭式→Laplace/MCMC/变分(后续)。
- **English**: Conjugate closed forms are special cases; complex models have no closed-form posterior → Laplace/MCMC/VI (coming up).
